# Research notebook
Run the bootstrap cell first. Review the experiment parameters and data paths before executing the remaining cells. Outputs are intentionally cleared for version control.


In [ ]:
from pathlib import Path
import os
import sys
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
os.chdir(project_root)
sys.path.insert(0, str(project_root / "src"))
Path("runs/notebooks").mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from remoma.graph.tmfg import TMFG

In [ ]:
df_nmi = pd.read_csv("data/similarity/lob_similarity_nmi_rellag_lag150_bins2000_mean.csv")

In [ ]:
df_nmi

In [ ]:
df_nmi.describe()

In [ ]:
df_nmi_fixed = df_nmi.set_index('Unnamed: 0')

In [ ]:
cols_lag0 = [c for c in df_nmi.columns if 'lag_0' in str(c)]

df_filtered = df_nmi_fixed.copy()

is_lag0_col = df_nmi_fixed.columns.isin(cols_lag0)
is_lag0_row = df_nmi_fixed.index.astype(str).str.contains('lag_0')

mask = np.outer(is_lag0_row, np.ones(len(is_lag0_col), dtype=bool)) | \
       np.outer(np.ones(len(is_lag0_row), dtype=bool), is_lag0_col)
df_filtered.values[~mask] = 0


In [ ]:
df_filtered.describe()

In [ ]:
tmfg = TMFG()
tmfg.fit(weights=df_filtered, output='unweighted_sparse_W_matrix')
cliques, separators, adj_matrix = tmfg.transform()

In [ ]:
df_adj = pd.DataFrame(adj_matrix, index=df_filtered.columns, columns=df_filtered.columns)

# Save the labeled adjacency as CSV.
df_adj.to_csv('runs/notebooks/tmfg_adj_matrix_cisco_2000_bins.csv')

In [ ]:
# Create the NetworkX graph.
G = nx.from_pandas_adjacency(pd.DataFrame(adj_matrix, index=df_filtered.index, columns=df_filtered.columns))

# Graph analysis

## Historical interpretation

The original run reported prominent ask/bid hubs, frequent separator nodes, and low variable diversity within cliques after suppressing redundant lag relations. These observations describe one filtered graph; they do not establish causal influence or independence between market processes.

Reported network statistics were 9,054 edges, mean clustering approximately 0.83, and mean shortest-path length approximately 5.65. Recompute these values for the supplied matrix. A small-world claim requires comparison with an appropriate null model.


In [ ]:
# Compute graph metrics.
degree_cent = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G)

# Build a comparison table.
df_stats = pd.DataFrame({
    'Degree Centrality': pd.Series(degree_cent),
    'Betweenness': pd.Series(betweenness)
}).sort_values(by='Degree Centrality', ascending=False)

print("Top 20 variables by degree centrality:")
print(df_stats.head(20))

# Plot the degree-centrality distribution.
plt.figure(figsize=(8, 4))
sns.histplot(df_stats['Degree Centrality'], bins=30, kde=True)
plt.title("Network Degree Centrality Distribution")
plt.show()

In [ ]:
# Inspect the first clique (initial graph core).
core_nodes = [df_nmi_fixed.columns[i] for i in cliques[0]]
print(f"Initial graph core (first 4-clique): {core_nodes}")

# Inspect clique composition.
def analyze_clique_composition(clique_indices):
    nodes = [df_nmi_fixed.columns[i] for i in clique_indices]
    base_names = [n.split('_lag')[0] for n in nodes]
    return len(set(base_names))

clique_diversity = [analyze_clique_composition(c) for c in cliques]
plt.figure(figsize=(8, 4))
sns.countplot(x=clique_diversity)
plt.title("Clique Diversity (Distinct Variables per Clique)")
plt.xlabel("Distinct variables in a four-node clique")
plt.show()

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

# --- 1. CHECK TOPOLOGICAL CONSISTENCY ---

node_count = len(G.nodes)
expected_edges = 3 * node_count - 6
observed_edges = len(G.edges)

print(f"--- Topological Validation ---")
print(f"Nodes: {node_count} | Observed edges: {observed_edges} | Expected: {expected_edges}")
if observed_edges == expected_edges:
    print("Edge count matches 3N - 6; this check alone does not prove planarity.")

# --- 2. IDENTIFY FREQUENT SEPARATOR NODES ---

sep_flat = [node_idx for s in separators for node_idx in s]
sep_counts = Counter(sep_flat)

# Convert node indices to variable names.
separator_names = {df_nmi_fixed.columns[idx]: count for idx, count in sep_counts.items()}
separator_counts = pd.Series(separator_names).sort_values(ascending=False)

print(f"\n--- Top 10 Separator Nodes (Occurrence Count) ---")
print(separator_counts.head(10))
#

# --- 3. CROSS-VARIABLE INTERACTION ANALYSIS ---

clique_diversity = []
for c in cliques:
    node_names = [df_nmi_fixed.columns[i] for i in c]
    base_variables = set([n.split('_lag')[0] for n in node_names])
    clique_diversity.append(len(base_variables))

diversity_counts = Counter(clique_diversity)
print(f"\n--- Clique Diversity ---")
for n_var, count in diversity_counts.items():
    print(f"Cliques with {n_var} distinct variables: {count}")

# --- 4. HIERARCHY ANALYSIS (DISTANCE FROM CORE) ---
core_node_idx = cliques[0][0] 
core_node_name = df_nmi_fixed.columns[core_node_idx]

print(f"Computing distances from core node: {core_node_name}")

# Compute shortest-path distance in edge hops.
distances = nx.single_source_shortest_path_length(G, core_node_name)
distance_series = pd.Series(distances).sort_values()

print(f"\n--- Nodes Closest to the Core (Topological Distance) ---")
# Show ten nodes after the first four sorted distance entries.
print(distance_series.iloc[4:14])

# --- 5. PATH LENGTH AND CLUSTERING ---
# Path length and clustering alone do not establish small-world structure; a reference network is required.

# Compute mean shortest-path length.
path_length = nx.average_shortest_path_length(G)

# Compute the mean clustering coefficient.
clustering_coeff = nx.average_clustering(G)

print(f"\n--- Path Length and Clustering ---")
print(f"Average Path Length: {path_length:.2f}")
print(f"Mean Clustering Coefficient: {clustering_coeff:.2f}")

In [ ]:
# G is the NetworkX graph constructed above.
ask_bid_edges = 0
ask_ask_edges = 0
bid_bid_edges = 0

# Iterate over graph edges.
for u, v in G.edges():
    # Classify the three side combinations.
    if ('ask' in u and 'bid' in v) or ('bid' in u and 'ask' in v):
        ask_bid_edges += 1
    elif 'ask' in u and 'ask' in v:
        ask_ask_edges += 1
    elif 'bid' in u and 'bid' in v:
        bid_bid_edges += 1

print(f"Total graph edges: {G.number_of_edges()}")
print(f"- Edges Ask-Ask:  {ask_ask_edges}")
print(f"- Edges Bid-Bid:  {bid_bid_edges}")
print(f"- Edges Ask-Bid:  {ask_bid_edges}")

# Plot the edge-type distribution.
plot_data = {
    'Ask-Ask': ask_ask_edges, 
    'Bid-Bid': bid_bid_edges, 
    'Ask-Bid (Cross)': ask_bid_edges
}

plt.figure(figsize=(8, 5))
sns.barplot(x=list(plot_data.keys()), y=list(plot_data.values()), palette="viridis")
plt.title("Network Interaction Distribution (NMI)")
plt.ylabel("Number of edges")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Draw the graph.

In [ ]:
plt.figure(figsize=(12, 12))
nx.draw(G, node_size=5, with_labels=False, edge_color='gray', alpha=0.5)
plt.show()

In [ ]:
from pyvis.network import Network
import matplotlib.colors as mcolors
import random


net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", select_menu=True)

base_variables = sorted(list(set([n.split('_lag')[0] for n in df_nmi_fixed.columns])))

color_palette = list(mcolors.TABLEAU_COLORS.values()) 
random.shuffle(color_palette)
color_map = {base: color_palette[i % len(color_palette)] for i, base in enumerate(base_variables)}

for node in G.nodes:
    base_name = node.split('_lag')[0]
    color = color_map.get(base_name, "#ffffff")
    net.add_node(node, label=node, color=color, title=f"Variable: {base_name}")

for edge in G.edges:
    net.add_edge(edge[0], edge[1])

net.set_options("""
{
  "physics": {
    "barnesHut": {
      "gravity": -80000,
      "damping": 0.5
    },
    "minVelocity": 0.75,
    "stabilization": {
      "enabled": true,
      "iterations": 2000
    }
  },
  "interaction": {
    "dragNodes": true,
    "hideEdgesOnDrag": false,
    "hideNodesOnDrag": false
  }
}
""")

net.save_graph("runs/notebooks/tmfg_network.html")